# Tracker BoT-SORT (Ultralytics) — detección de movimiento desde dron

**Qué hace.** Mismo sistema que el tracker propio (YOLO + cambios de imagen sin clase → tracking → ¿se mueve de verdad?), pero la asociación entre frames la hace **BoT-SORT real de Ultralytics**, alimentado con *nuestras* detecciones.

**Qué lo distingue.** Filtro de Kalman + asociación en dos pasadas (como ByteTrack) + **GMC** (*global motion compensation*): estima el movimiento de la cámara con su propio método (`sparseOptFlow` por defecto) y corrige la predicción del Kalman. ReID desactivado (no requiere pesos extra). Con `GMC_METHOD = "none"` queda como BoT-SORT sin compensación, útil para aislar el efecto del GMC.

Notebook **autocontenido**: el motor común es idéntico al de `tracker_propio.ipynb` y `comparativa_trackers_v8.ipynb`; solo cambia la PARTE 3. `TrackBook` convierte la salida de Ultralytics en nuestros `Track`, así la decisión de movimiento y el dibujo son exactamente los mismos.

## Orden de ejecución
`PARTE 1` → … → `PARTE 5` (solo definen cosas) → `PARTE 6` (paso A: verificar Ultralytics; paso B: procesar).

In [ ]:
# ======================================================================
# IMPORTACIONES
# Qué hace : Carga las librerías
# ¿Se toca? : No
# ======================================================================
import csv
import os
from collections import deque, defaultdict
from dataclasses import dataclass

import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

## PARTE 1 — Configuración (lo único que se ajusta)
Todos los parámetros están agrupados por etiqueta: `[CAMARA]`, `[TRACKER PROPIO]`, `[MOVIMIENTO]`, `[PROPUESTAS]`, `[CLASE]`, `[PARALAJE]`, `[FUSION]`. Los marcados *igual para TODOS los trackers* afectan a la comparativa completa. Las distancias están en píxeles de la resolución de trabajo (`work_width`).

In [ ]:
# ======================================================================
# PARTE 1 · Config
# Qué hace : Todos los parámetros del sistema
# ¿Se toca? : SÍ — aquí se ajusta todo
# ======================================================================
@dataclass
class Config:
    work_width: int = 960          # ancho al que se estima el movimiento global y se hace el tracking
    out_width: int = 1920          # ancho del video de salida (0 = resolución original)
    # [CAMARA] movimiento global de la cámara (homografía)
    min_inliers: int = 30          # por debajo, la homografía se considera poco fiable
    ransac_px: float = 1.5         # tolerancia RANSAC: más baja = se ajusta más al plano del suelo
    grid: tuple = (4, 3)           # (cols, filas): reparte los puntos por toda la imagen (evita que domine una zona)
    # [ASOCIACION] umbrales que se le pasan al tracker de Ultralytics
    conf_high: float = 0.35        # detecciones "fuertes" (1ª asociación, crean tracks nuevos)
    conf_low: float = 0.10         # detecciones "débiles" (2ª asociación, estilo ByteTrack)
    max_miss_sec: float = 1.0      # segundos que un track sobrevive sin detección
    min_hits: int = 4              # detecciones para confirmar un track
    # [MOVIMIENTO] cuándo un track cuenta como 'en movimiento' (igual para TODOS los trackers)
    window_sec: float = 1.0        # ventana temporal para medir el desplazamiento neto
    min_move_px: float = 6.0       # desplazamiento mínimo (en la ventana) para considerar movimiento
    rel_move: float = 0.30         # ...y además >= rel_move * tamaño del objeto (sqrt(ancho*alto)); frena cajas grandes inestables
    adaptive: bool = False         # umbral adaptativo con la población de tracks (solo si la mayoría es estática)
    max_adaptive_px: float = 12.0  # tope del umbral adaptativo
    mad_k: float = 4.0             # umbral adaptativo = mediana + k*MAD de la población de tracks
    max_box_area_frac: float = 0.03  # se descartan detecciones con caja mayor a esta fracción del frame
    on_frames: int = 3             # evaluaciones seguidas "moviéndose" para activar el estado
    off_frames: int = 10           # evaluaciones seguidas "quieto" para desactivarlo
    # [PROPUESTAS] cambios de imagen sin clase: objetos que YOLO no conoce (igual para TODOS los trackers)
    use_proposals: bool = True
    proposal_gap_sec: float = 0.15 # separación temporal entre los frames que se restan
    diff_thr: int = 25
    min_area: int = 40
    max_area: int = 6000
    dilate: int = 5
    border: int = 15
    max_aspect: float = 5.0
    max_residual: float = 3.5      # diferencia media tras alinear; si se supera, se omiten propuestas
    proposal_conf: float = 0.5
    max_coherence: float = 0.80    # estructura de una sola orientación (cables, líneas, bordillos) -> no es un objeto
    min_straightness: float = 0.60 # desplazamiento neto / recorrido en la ventana; un objeto real avanza en línea, el ruido deambula
    # [CLASE] evita que una detección suelta de baja confianza cambie la etiqueta de un track
    cls_min_conf: float = 0.40     # solo las detecciones con esta confianza votan por una clase
    cls_min_hits: int = 3          # votos mínimos para que el track adopte una clase; si no, queda "desconocido"
    # [PARALAJE] lo que sobresale del suelo se desplaza respecto a él cuando el dron se traslada
    parallax_filter: bool = True
    parallax_mode: str = "temporal"    # "temporal" (v8, recomendado) | "ratio" (criterio del v7, para comparar)
    parallax_angle_deg: float = 30.0   # el residuo debe ser paralelo al flujo del suelo en ese punto
    parallax_ratio: float = 0.15       # [solo "ratio"] h/Z máximo; con el dron bajo (poste) NUNCA se cumple
    parallax_k_min: float = -1.5       # [solo "temporal"] k<0: algo fijo en la IMAGEN (cable que resbala, borde)
    parallax_k_max: float = 3.0        # [solo "temporal"] k>0: paralaje; en input1 el poste llega a k≈2.5
    parallax_min_fit: float = 0.6      # [solo "temporal"] R² mínimo del ajuste  residuo(t) ≈ k·flujo_suelo_acumulado(t)
    parallax_min_ground_px: float = 3.0  # [solo "temporal"] flujo NETO del suelo mínimo en la ventana (en hover no se decide)
    parallax_skip: tuple = ("car", "truck", "bus", "motorcycle", "small vehicle", "large vehicle")  # clases que NO se filtran
    show_parallax: bool = True     # dibujar en gris lo descartado por paralaje
    id_badge: bool = True          # dibuja el ID en un recuadro de color (cada ID, un color): los cambios de ID se ven
    debug_residual: bool = False   # guarda un segundo video con el residuo (frame - frame previo compensado) y las cajas
    # [FUSION] fragmentos que se mueven juntos = una sola caja (vehículos largos)
    merge_motion: bool = True
    merge_gap: float = 0.6         # separación máxima entre cajas, como fracción del lado mayor
    merge_angle_deg: float = 25.0  # diferencia máxima de dirección
    merge_speed_ratio: float = 0.35  # diferencia relativa máxima de velocidad
    merge_max_aspect: float = 4.0  # si el grupo fusionado sería más alargado que esto (y sin clase), NO se fusiona
    max_unknown_aspect: float = 6.0  # un track SIN clase más alargado que esto es una estructura (cable, borde), no un objeto

## PARTE 2 — Motor compartido (igual para los tres trackers)
### 2.1 Movimiento de la cámara
**Entra:** dos frames consecutivos. **Sale:** la matriz `H` que explica cómo se movió la cámara (homografía). Con `H` se pueden expresar las posiciones respecto al suelo y no a la pantalla.

In [ ]:
# ======================================================================
# 2.1 · GlobalMotion
# Qué hace : Estima el movimiento de la cámara entre frames (LK + RANSAC)
# ¿Se toca? : No
# ======================================================================
def warp_pts(H, pts):
    pts = np.asarray(pts, np.float32).reshape(-1, 2)
    if len(pts) == 0:
        return pts
    return cv2.perspectiveTransform(pts.reshape(-1, 1, 2), H).reshape(-1, 2)


class GlobalMotion:
    """Estima la homografía H (frame previo -> frame actual) con LK + RANSAC."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.prev = None

    def _features(self, gray):
        """Puntos repartidos en una rejilla, para que ninguna zona (techos, pastizal) domine la homografía."""
        h, w = gray.shape
        gx, gy = self.cfg.grid
        pts = []
        for i in range(gy):
            for j in range(gx):
                y0, y1, x0, x1 = i * h // gy, (i + 1) * h // gy, j * w // gx, (j + 1) * w // gx
                p = cv2.goodFeaturesToTrack(gray[y0:y1, x0:x1], 150, 0.01, 8)
                if p is not None:
                    p = p.reshape(-1, 2) + np.array([x0, y0], np.float32)
                    pts.append(p)
        return np.vstack(pts).reshape(-1, 1, 2).astype(np.float32) if pts else None

    def step(self, gray):
        if self.prev is None:
            self.prev = gray
            return np.eye(3), True
        H, ok = np.eye(3), False
        p0 = self._features(self.prev)
        if p0 is not None and len(p0) >= 10:
            p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, p0, None, winSize=(21, 21), maxLevel=3)
            good = st.ravel() == 1
            if good.sum() >= 10:
                Hc, inl = cv2.findHomography(p0[good], p1[good], cv2.RANSAC, self.cfg.ransac_px)
                if Hc is not None and inl.sum() >= self.cfg.min_inliers:
                    H, ok = Hc, True
        self.prev = gray
        return H, ok

### 2.2 Detectores
**Entra:** un frame. **Sale:** cajas `[x1,y1,x2,y2,conf,clase]`. `YoloDetector` solo detecta (no hace tracking). Cualquier otro detector (DETR…) se conecta devolviendo ese mismo formato.

In [ ]:
# ======================================================================
# 2.2 · YoloDetector
# Qué hace : Detecta objetos con un modelo YOLO (incluye modelos aéreos OBB)
# ¿Se toca? : Solo sus parámetros al crearlo (modelo, imgsz, conf)
# ======================================================================
def resolve_class_ids(names, wanted):
    """Convierte nombres de clase (p. ej. "small vehicle") en los ids del modelo. names = model.names."""
    inv = {str(v).lower(): k for k, v in names.items()}
    ids, missing = [], []
    for w in wanted:
        (ids if str(w).lower() in inv else missing).append(inv.get(str(w).lower(), w))
    if missing:
        raise ValueError(f"Clases que no existen en el modelo: {missing}. Disponibles: {sorted(inv)}")
    return ids


class YoloDetector:
    """Solo DETECTA (predict), no hace tracking. El tracking lo hace nuestro Tracker."""

    def __init__(self, weights="yolo11n.pt", imgsz=1280, conf=0.10, classes=None,
                 device=None, tiles=None, overlap=0.2, class_names=None):
        from ultralytics import YOLO
        self.model = YOLO(weights)
        self.names = self.model.names
        if class_names:                              # filtrar por NOMBRE: no hace falta saber los ids del modelo
            classes = resolve_class_ids(self.names, class_names)
        self.imgsz, self.conf, self.classes, self.device = imgsz, conf, classes, device
        self.tiles, self.overlap = tiles, overlap      # tiles=(filas, columnas) o None

    def _predict(self, img):
        r = self.model.predict(img, imgsz=self.imgsz, conf=self.conf, classes=self.classes,
                               device=self.device, verbose=False)[0]
        b = getattr(r, "obb", None)          # modelos *-obb (DOTA): cajas orientadas; se usa su caja horizontal
        if b is None:
            b = r.boxes
        if b is None or len(b) == 0:
            return np.zeros((0, 6), np.float32)
        return np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None],
                               b.cls.cpu().numpy()[:, None]], 1).astype(np.float32)

    def __call__(self, frame_bgr, scale):
        if not self.tiles:
            d = self._predict(frame_bgr)
        else:
            import torch, torchvision
            H, W = frame_bgr.shape[:2]
            rows, cols = self.tiles
            th, tw = H / rows, W / cols
            outs = []
            for i in range(rows):
                for j in range(cols):
                    y0, y1 = int(max(0, i * th - self.overlap * th)), int(min(H, (i + 1) * th + self.overlap * th))
                    x0, x1 = int(max(0, j * tw - self.overlap * tw)), int(min(W, (j + 1) * tw + self.overlap * tw))
                    t = self._predict(frame_bgr[y0:y1, x0:x1])
                    t[:, [0, 2]] += x0
                    t[:, [1, 3]] += y0
                    outs.append(t)
            d = np.concatenate(outs, 0) if outs else np.zeros((0, 6), np.float32)
            if len(d):
                keep = torchvision.ops.batched_nms(torch.from_numpy(d[:, :4]), torch.from_numpy(d[:, 4]),
                                                   torch.from_numpy(d[:, 5]).long(), 0.5).numpy()
                d = d[keep]
        d[:, :4] *= scale          # a coordenadas de trabajo
        return d

### 2.3 Propuestas de movimiento (objetos que YOLO no conoce)
Diferencia entre el frame actual y uno anterior ya compensado por `H`. Genera cajas **sin clase** (`desconocido`) para cosas en movimiento que ningún modelo etiqueta (paracaídas, peatones vistos desde arriba).

In [ ]:
# ======================================================================
# 2.3 · MotionProposals
# Qué hace : Cajas sin clase donde algo cambia tras compensar la cámara
# ¿Se toca? : No (parámetros en Config → [PROPUESTAS])
# ======================================================================
class MotionProposals:
    """Cambios respecto al frame t-gap, ya compensando la cámara. Sirven para objetos que YOLO no conoce."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        self.gap = max(2, int(round(cfg.proposal_gap_sec * fps)))
        self.grays = deque(maxlen=self.gap + 1)
        self.Hs = deque(maxlen=self.gap)
        self.oks = deque(maxlen=self.gap)
        self.last_diff = None

    def step(self, gray, H, ok):
        c = self.cfg
        if len(self.grays) > 0:
            self.Hs.append(H)
            self.oks.append(ok)
        self.grays.append(gray)
        if len(self.grays) < self.gap + 1 or not all(self.oks):
            return np.zeros((0, 6), np.float32)
        M = np.eye(3)
        for h in self.Hs:                       # H_t @ ... @ H_{t-gap+1}
            M = h @ M
        prev, curr = self.grays[0], gray
        h_, w_ = curr.shape
        warped = cv2.warpPerspective(prev, M, (w_, h_))
        valid = cv2.erode(cv2.warpPerspective(np.full((h_, w_), 255, np.uint8), M, (w_, h_)),
                          np.ones((11, 11), np.uint8))
        d = cv2.bitwise_and(cv2.absdiff(curr, warped), valid)
        self.last_diff = d
        if valid.any() and d[valid > 0].mean() > c.max_residual:
            return np.zeros((0, 6), np.float32)
        m = (d > c.diff_thr).astype(np.uint8) * 255
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        m = cv2.dilate(m, np.ones((c.dilate, c.dilate), np.uint8))
        n, _, s, _ = cv2.connectedComponentsWithStats(m)
        gx = cv2.Sobel(curr, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(curr, cv2.CV_32F, 0, 1, ksize=3)
        out = []
        for k in range(1, n):
            x, y, bw, bh, a = s[k]
            if not (c.min_area <= a <= c.max_area):
                continue
            if x < c.border or y < c.border or x + bw > w_ - c.border or y + bh > h_ - c.border:
                continue
            ar = bw / max(bh, 1)
            if ar > c.max_aspect or ar < 1 / c.max_aspect:
                continue
            sx, sy = gx[y:y + bh, x:x + bw], gy[y:y + bh, x:x + bw]
            jxx, jyy, jxy = float((sx * sx).sum()), float((sy * sy).sum()), float((sx * sy).sum())
            coh = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-6)
            if coh > c.max_coherence:
                continue                           # línea/cable/borde: el centroide del cambio resbala a lo largo (apertura)
            out.append([x, y, x + bw, y + bh, c.proposal_conf, -1])
        return np.array(out, np.float32).reshape(-1, 6)

## PARTE 3 — El tracker
### 3.1 Track (común)
Historial de centros **en coordenadas estabilizadas** (se re-proyecta con `H` cada frame), votos de clase y estado de movimiento. Aunque BoT-SORT no use `H` para asociar, la *decisión de movimiento* sí se mide en el espacio estabilizado, igual que en el propio.
### 3.2 BoT-SORT de Ultralytics + TrackBook
Se usa el **algoritmo real de Ultralytics**, alimentado con **nuestras** detecciones. `TrackBook` convierte su salida (ids + cajas) a nuestros objetos `Track`.
- Es un algoritmo, no un modelo entrenado (ReID desactivado).
- `selftest_ultralytics()` (PARTE 6, paso A) comprueba que tu versión responde como se espera.

In [ ]:
# ======================================================================
# 3.1 · iou_pad + Track
# Qué hace : Objeto 'track' común: historial estabilizado, votos de clase, estado de movimiento
# ¿Se toca? : No (parámetros en Config → [TRACKER PROPIO])
# ======================================================================
def iou_pad(a, b, pad):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = a + np.array([-pad, -pad, pad, pad]); b = b + np.array([-pad, -pad, pad, pad])
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)


class Track:
    def __init__(self, tid, det, fidx, maxlen):
        self.id = tid
        self.box = det[:4].astype(np.float32).copy()
        self.hits, self.miss = 1, 0
        self.votes = defaultdict(float)
        self.counts = defaultdict(int)
        self._vote(det)
        self.pts = np.array([self._center()], np.float32)   # historial de centros, siempre en coords del frame ACTUAL
        self.gs = np.zeros((1, 2), np.float32)             # flujo que la cámara induce en el track (H·c - c), por frame
        self._g = np.zeros(2, np.float32)
        self.par, self.par_frames = False, 0
        self.straight = 1.0
        self.toggles = 0
        self.maxlen = maxlen
        self.k_par = None                                   # k del ajuste de paralaje (diagnóstico)
        self.first, self.last = fidx, fidx
        self.disp, self.moving = 0.0, False
        self.vec = np.zeros(2, np.float32)
        self.on_cnt = self.off_cnt = 0
        self.moving_frames = 0
        self.max_disp = 0.0

    def _center(self):
        return [(self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2]

    min_conf, min_hits = 0.40, 3                    # se fijan desde Config en Tracker.__init__

    def _vote(self, det):
        if det[5] >= 0 and det[4] >= Track.min_conf:
            self.votes[int(det[5])] += float(det[4]); self.counts[int(det[5])] += 1

    @property
    def cls(self):
        if not self.votes:
            return -1
        c = max(self.votes, key=self.votes.get)
        return c if self.counts[c] >= Track.min_hits else -1

    def warp(self, H):
        old_c = np.array(self._center(), np.float32)
        c = warp_pts(H, [self.box[:2], self.box[2:]])
        self.box = c.reshape(-1).astype(np.float32)
        self._g = (np.array(self._center(), np.float32) - old_c)        # flujo del suelo en este punto
        self.pts = warp_pts(H, self.pts)
        A = (H[:2, :2] / H[2, 2]).astype(np.float32)
        self.gs = self.gs @ A.T                                         # llevar el historial al frame actual

    def update(self, det, fidx):
        self.box = det[:4].astype(np.float32).copy()
        self.hits += 1; self.miss = 0; self.last = fidx
        self._vote(det)
        self.pts = np.vstack([self.pts, self._center()])[-self.maxlen:]
        self.gs = np.vstack([self.gs, self._g])[-self.maxlen:]

In [ ]:
# ======================================================================
# 3.2 · UltralyticsTracker + TrackBook
# Qué hace : ByteTrack / BoT-SORT reales + conversión a nuestros Track
# ¿Se toca? : No
# ======================================================================
class _Dets:
    """Imita lo mínimo del objeto Boxes de Ultralytics que consumen sus trackers (xyxy, xywh, conf, cls, indexado)."""

    def __init__(self, xyxy, conf, cls):
        self.xyxy = np.asarray(xyxy, np.float32).reshape(-1, 4)
        self.conf = np.asarray(conf, np.float32).reshape(-1)
        self.cls = np.asarray(cls, np.float32).reshape(-1)
        x1, y1, x2, y2 = self.xyxy.T if len(self.xyxy) else (np.zeros(0),) * 4
        self.xywh = np.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], 1).astype(np.float32) if len(self.xyxy) else np.zeros((0, 4), np.float32)
        self.data = np.concatenate([self.xyxy, self.conf[:, None], self.cls[:, None]], 1)

    def __len__(self):
        return len(self.conf)

    def __getitem__(self, idx):
        return _Dets(self.xyxy[idx], self.conf[idx], self.cls[idx])


from types import SimpleNamespace


class _NS(SimpleNamespace):
    """Respaldo de IterableSimpleNamespace por si esta versión de Ultralytics ya no lo exporta donde se esperaba."""

    def get(self, k, default=None):
        return getattr(self, k, default)

    def __iter__(self):
        return iter(vars(self).items())


def _ul_import():
    """Importa los trackers de Ultralytics probando varias ubicaciones (la API cambia entre versiones)."""
    import importlib
    ul = importlib.import_module("ultralytics")
    errs, byte, bot, ns = [], None, None, None
    for mb, mt in (("ultralytics.trackers", "ultralytics.trackers"),
                   ("ultralytics.trackers.byte_tracker", "ultralytics.trackers.bot_sort")):
        try:
            byte = getattr(importlib.import_module(mb), "BYTETracker")
            bot = getattr(importlib.import_module(mt), "BOTSORT")
            break
        except Exception as e:
            errs.append(f"{mb}: {e}")
    if byte is None:
        raise ImportError("No encuentro BYTETracker/BOTSORT en Ultralytics. Intentos: " + " | ".join(errs))
    for m in ("ultralytics.utils", "ultralytics.cfg", "ultralytics.utils.checks"):
        try:
            ns = getattr(importlib.import_module(m), "IterableSimpleNamespace")
            break
        except Exception:
            continue
    return ul, byte, bot, (ns or _NS)


def _ul_load_yaml(ul, fname):
    """Lee bytetrack.yaml / botsort.yaml directamente de la carpeta del paquete (sin yaml_load ni check_yaml)."""
    import yaml
    pkg = os.path.dirname(ul.__file__)
    path = os.path.join(pkg, "cfg", "trackers", fname)
    if not os.path.isfile(path):
        for root, _, files in os.walk(pkg):
            if fname in files:
                path = os.path.join(root, fname)
                break
        else:
            raise FileNotFoundError(f"No encuentro {fname} dentro de {pkg}")
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def diagnostico_ultralytics():
    """Imprime lo necesario para adaptar el código si tu versión de Ultralytics difiere de lo esperado."""
    import importlib, inspect
    ul = importlib.import_module("ultralytics")
    print("ultralytics", getattr(ul, "__version__", "?"), "|", os.path.dirname(ul.__file__))
    tdir = os.path.join(os.path.dirname(ul.__file__), "cfg", "trackers")
    print("YAML de trackers:", sorted(os.listdir(tdir)) if os.path.isdir(tdir) else "carpeta cfg/trackers NO existe")
    try:
        u = importlib.import_module("ultralytics.utils")
        print("nombres con 'yaml' en ultralytics.utils:", [n for n in dir(u) if "yaml" in n.lower()])
    except Exception as e:
        print("ultralytics.utils:", e)
    try:
        _, byte, bot, _ns = _ul_import()
        for nm, cl in (("BYTETracker", byte), ("BOTSORT", bot)):
            print(nm, "__init__", inspect.signature(cl.__init__), "| update", inspect.signature(cl.update))
    except Exception as e:
        print("Trackers:", e)


class UltralyticsTracker:
    """ByteTrack o BoT-SORT reales de Ultralytics, alimentados con NUESTRAS detecciones (así la comparativa es justa).
    kind: "bytetrack" | "botsort".  gmc_method (solo botsort): "sparseOptFlow" | "orb" | "ecc" | "none"."""

    FILES = {"bytetrack": "bytetrack.yaml", "botsort": "botsort.yaml"}

    def __init__(self, kind, cfg, fps, gmc_method=None, overrides=None):
        ul, BYTE, BOT, NS = _ul_import()
        args = NS(**_ul_load_yaml(ul, self.FILES[kind]))
        args.track_high_thresh = cfg.conf_high
        args.track_low_thresh = cfg.conf_low
        args.new_track_thresh = cfg.conf_high
        args.track_buffer = int(round(cfg.max_miss_sec * 30))     # Ultralytics lo reescala con frame_rate/30
        if kind == "botsort":
            args.with_reid = False                                 # sin ReID: no requiere pesos adicionales
            if gmc_method is not None:
                args.gmc_method = gmc_method
        for k, v in (overrides or {}).items():
            setattr(args, k, v)
        self.kind, self.args = kind, args
        Cls = BYTE if kind == "bytetrack" else BOT
        self.tracker = self._build(Cls, args, fps)
        # Ventana de pérdida: se fija explícitamente en FRAMES para que no dependa de cómo cada versión
        # reescala track_buffer con el frame rate (en algunas versiones ya ni existe ese parámetro).
        target = max(2, int(round(cfg.max_miss_sec * fps)))
        for attr in ("max_time_lost", "buffer_size"):
            if hasattr(self.tracker, attr):
                setattr(self.tracker, attr, target)
        self.target_lost_frames = target

    @staticmethod
    def _build(Cls, args, fps):
        """Construye el tracker según la firma REAL del constructor (cambia entre versiones de Ultralytics)."""
        import inspect
        params = [p for n, p in inspect.signature(Cls.__init__).parameters.items() if n != "self"]
        names = [p.name for p in params]
        kw = {}
        if "args" in names:
            kw["args"] = args
        for n in ("frame_rate", "fps"):
            if n in names:
                kw[n] = int(round(fps))
        if kw:
            return Cls(**kw)
        return Cls(args)                       # primer parámetro posicional (nombre distinto de "args")

    def update(self, dets, img):
        """dets: Nx6 [x1,y1,x2,y2,conf,cls] en coordenadas de trabajo; img: frame BGR de trabajo (lo usa el GMC)."""
        d = _Dets(dets[:, :4], dets[:, 4], dets[:, 5]) if len(dets) else _Dets([], [], [])
        try:
            out = self.tracker.update(d, img)
        except TypeError:
            out = self.tracker.update(d)
        out = np.asarray(out, np.float32)
        return out[:, :7] if out.ndim == 2 and len(out) else np.zeros((0, 7), np.float32)   # x1,y1,x2,y2,id,conf,cls


def selftest_ultralytics(cfg=None, fps=30.0):
    """Ejecútala ANTES de procesar: comprueba que botsort de Ultralytics responde como espera este código."""
    import inspect
    cfg = cfg or Config()
    img = (np.random.default_rng(0).random((540, 960, 3)) * 255).astype(np.uint8)
    d = np.array([[100, 100, 140, 160, 0.9, 2], [300, 200, 360, 260, 0.8, 0]], np.float32)
    try:
        t = UltralyticsTracker("botsort", cfg, fps, gmc_method=GMC_METHOD)
        for _ in range(3):
            out = t.update(d, img)
        print(f"botsort OK | salida {out.shape} | ids {out[:, 4].astype(int).tolist()}")
        print(f"  __init__{inspect.signature(type(t.tracker).__init__)}")
        print("  ventana de pérdida (frames):", {a: getattr(t.tracker, a) for a in ("max_frames_lost", "max_time_lost", "buffer_size") if hasattr(t.tracker, a)}
              or "no encontrada")
        if "botsort" == "botsort":
            try:
                w = t.tracker.gmc.apply(img, d[:, :4])
                print("  GMC OK | warp =", np.round(np.asarray(w), 3).tolist())
            except Exception as e:
                print("  GMC FALLA (Ultralytics lo ignora en silencio y no compensaría la cámara):", e)
    except Exception as e:
        print("FALLO:", type(e).__name__, e, "\n--- diagnóstico (mándame esto) ---")
        diagnostico_ultralytics()


class TrackBook:
    """Convierte la salida de un tracker externo (ids + cajas) en objetos Track propios, para reutilizar
    EXACTAMENTE la misma decisión de movimiento (MotionState) y el mismo dibujo en todos los trackers."""

    def __init__(self, cfg, fps):
        self.cfg = cfg
        Track.min_conf, Track.min_hits = cfg.cls_min_conf, cfg.cls_min_hits
        self.tracks = {}
        self.max_miss = max(2, int(round(cfg.max_miss_sec * fps)))
        self.maxlen = int(round(cfg.window_sec * fps)) + 1

    def update(self, out, H, H_ok, fidx):
        for t in self.tracks.values():
            t.warp(H)
            if not H_ok:
                t.pts = t.pts[-1:]; t.gs = t.gs[-1:]
        seen = set()
        for r in out:
            tid = int(r[4])
            det = np.array([r[0], r[1], r[2], r[3], r[5], r[6]], np.float32)
            if tid in self.tracks:
                self.tracks[tid].update(det, fidx)
            else:
                self.tracks[tid] = Track(tid, det, fidx, self.maxlen)
            seen.add(tid)
        for tid in list(self.tracks):
            if tid not in seen:
                self.tracks[tid].miss += 1
                if self.tracks[tid].miss > self.max_miss:
                    del self.tracks[tid]
        return [t for t in self.tracks.values() if t.hits >= self.cfg.min_hits and t.miss == 0]

In [ ]:
# ======================================================================
# 3.3 · TrackerSistema (BOTSORT)
# Qué hace : Interfaz común que usa run(): update(...) -> tracks confirmados
# ¿Se toca? : No
# ======================================================================
class TrackerSistema:
    nombre, titulo = "botsort", "BOTSORT - con GMC (compensa camara)"

    def __init__(self, cfg, fps, gmc_method='sparseOptFlow'):
        self.ext = UltralyticsTracker("botsort", cfg, fps, gmc_method=gmc_method)
        self.book = TrackBook(cfg, fps)
        self.raw = set()

    def update(self, dets, H, H_ok, fidx, img):
        out = self.ext.update(dets, img)           # img = frame de trabajo (BGR); lo usa el GMC
        self.raw.update(int(x) for x in out[:, 4])
        return self.book.update(out, H, H_ok, fidx)

    @property
    def ids_emitidos(self):
        return len(self.raw)                       # IDs distintos emitidos por el tracker

## PARTE 4 — Decisión de movimiento y dibujo (igual para los tres)
### 4.1 ¿Se mueve o no?
Mide el **desplazamiento neto** del track en una ventana (por defecto 1 s) **en coordenadas estabilizadas**, con tres defensas contra falsos positivos: exige avanzar en línea (`min_straightness`), descarta lo explicable por **paralaje** (v8: criterio *temporal*, ver abajo), y escala el umbral con el tamaño del objeto.

**Filtro de paralaje v8 (`parallax_mode="temporal"`).** En `input1.mp4` el dron *desciende* sobre un poste desde ~25 s. El criterio del v7 (`|residuo| ≤ 0.15·|flujo del suelo|`) no se activó **ni una vez** (`frames_paralaje = 0` en los tres CSV), porque con el dron tan bajo el poste tiene k = h/Z entre 0.3 y 2.5. Ahora se pide que el desplazamiento del track *siga en el tiempo* a k·(flujo acumulado del suelo), con k en `[parallax_k_min, parallax_k_max]`. En el tramo final (frames 1500-2568, solo propuestas) los frames marcados como en movimiento bajaron de **3144 a 1054** sin perder los vehículos reales del inicio (k = 12-49). `parallax_mode="ratio"` recupera el comportamiento del v7.

In [ ]:
# ======================================================================
# 4.1 · MotionState
# Qué hace : Decide si cada track está en movimiento
# ¿Se toca? : No (parámetros en Config → [MOVIMIENTO], [PARALAJE])
# ======================================================================
class MotionState:
    def __init__(self, cfg, fps, skip_ids=()):
        self.cfg = cfg
        self.W = int(round(cfg.window_sec * fps))
        self.thr = cfg.min_move_px
        self.skip_ids = set(skip_ids)

    @staticmethod
    def _straightness(p):
        """Desplazamiento neto / recorrido, sobre 6 tramos (medianas) para no contar el jitter de la caja."""
        if len(p) < 6:
            return 1.0
        m = np.array([np.median(ch, 0) for ch in np.array_split(p, 6)])
        path = float(np.linalg.norm(np.diff(m, axis=0), axis=1).sum())
        return float(np.linalg.norm(m[-1] - m[0]) / (path + 1e-6))

    def _parallax_like(self, t, n):
        """¿El desplazamiento del track lo explica la cámara y no un movimiento propio?"""
        if self.cfg.parallax_mode == "ratio":
            return self._parallax_ratio(t, n)
        return self._parallax_temporal(t, n)

    def _parallax_ratio(self, t, n):
        """Criterio del v7. Un objeto estático por encima del suelo deja un residuo PARALELO (y del mismo sentido) al flujo
        que la cámara induce en el suelo, con razón h/Z = |residuo| / |flujo| <= parallax_ratio.
        Problema detectado en input1: cuando el dron DESCIENDE sobre el poste, h/Z llega a 0.3-2.5 y nunca se cumple."""
        c = self.cfg
        g = t.gs[-n:]
        G = g.sum(0); Gn = float(np.linalg.norm(G))   # recorrido NETO del suelo (el ruido de H en hover no cuenta)
        if Gn < 1e-6 or t.disp < 1e-6:
            return False
        cos = float(t.vec @ G) / (t.disp * Gn)
        return cos >= np.cos(np.deg2rad(c.parallax_angle_deg)) and t.disp <= c.parallax_ratio * Gn

    def _parallax_temporal(self, t, n):
        """Criterio v8. Si el objeto está quieto en el mundo, su desplazamiento estabilizado P(t) es, frame a frame,
        PROPORCIONAL al flujo acumulado del suelo en ese punto C(t):  P(t) ≈ k·C(t).
          k > 0  -> paralaje: sobresale del suelo (poste, luminaria, árbol). k = h/(Z-h), sin tope pequeño.
          k ≈ -1 -> fijo en la IMAGEN, no en el suelo (tramo de cable que 'resbala', borde, artefacto de lente).
        Se exige: dirección paralela (±parallax_angle_deg), k en [k_min, k_max] y buen ajuste en el TIEMPO (R²).
        Un vehículo real tiene k >> k_max (en input1, 12-49) o no sigue el patrón del suelo.
        Límite: un peatón que camine justo en dirección radial mientras el dron baja puede confundirse."""
        c = self.cfg
        p, g = t.pts[-n:], t.gs[-n:]
        if len(p) < 6 or t.disp < 1e-6:
            t.k_par = None
            return False
        P = p - p[0]
        C = np.cumsum(g, 0); C = C - C[0]
        Gn = float(np.linalg.norm(C[-1]))
        if Gn < c.parallax_min_ground_px:
            t.k_par = None
            return False
        cos = float(t.vec @ C[-1]) / (t.disp * Gn)
        k = float((P * C).sum() / ((C * C).sum() + 1e-9))
        fit = 1.0 - float(((P - k * C) ** 2).sum()) / (float((P ** 2).sum()) + 1e-9)
        t.k_par = k
        return (abs(cos) >= np.cos(np.deg2rad(c.parallax_angle_deg))
                and c.parallax_k_min <= k <= c.parallax_k_max and fit >= c.parallax_min_fit)

    def update(self, tracks, H_ok):
        c = self.cfg
        if not H_ok:
            return
        disps = []
        for t in tracks:
            n = min(len(t.pts), self.W + 1)
            if n < max(5, int(0.5 * self.W)):
                t.disp = 0.0; t.vec = np.zeros(2, np.float32); t.par = False; continue
            p = t.pts[-n:]; k = max(1, n // 5)
            t.vec = (np.median(p[-k:], 0) - np.median(p[:k], 0)).astype(np.float32)
            t.disp = float(np.linalg.norm(t.vec))
            disps.append(t.disp)
        self.thr = c.min_move_px
        if c.adaptive and len(disps) >= 5:         # solo útil si la mayoría de los tracks son estáticos
            med = np.median(disps); mad = np.median(np.abs(np.array(disps) - med)) * 1.4826
            self.thr = float(min(max(c.min_move_px, med + c.mad_k * mad), c.max_adaptive_px))
        for t in tracks:
            n = min(len(t.pts), self.W + 1)
            if len(t.pts) < max(5, int(0.5 * self.W)):
                continue
            t.straight = self._straightness(t.pts[-n:])
            size = float(np.sqrt(max((t.box[2] - t.box[0]) * (t.box[3] - t.box[1]), 1.0)))
            thr_t = max(self.thr, c.rel_move * size)   # un objeto grande necesita moverse más para contar
            bw, bh = t.box[2] - t.box[0], t.box[3] - t.box[1]
            is_structure = t.cls < 0 and max(bw, bh) / max(min(bw, bh), 1.0) > c.max_unknown_aspect
            raw = t.disp > thr_t and t.straight >= c.min_straightness and not is_structure
            t.par = False
            if raw and c.parallax_filter and t.cls not in self.skip_ids and self._parallax_like(t, n):
                raw, t.par = False, True               # explicable por paralaje: no cuenta como movimiento
                t.par_frames += 1
            if raw:
                t.on_cnt += 1; t.off_cnt = 0
            else:
                t.off_cnt += 1; t.on_cnt = 0
            if not t.moving and t.on_cnt >= c.on_frames:
                t.moving = True; t.toggles += 1
            elif t.moving and t.off_cnt >= c.off_frames:
                t.moving = False; t.toggles += 1
            t.moving_frames += int(t.moving)
            t.max_disp = max(t.max_disp, t.disp)

### 4.2 Dibujo
**Verde** = estático · **Rojo** = en movimiento con clase · **Naranja** = en movimiento sin clase · **Gris** = descartado por paralaje. El **recuadro de color con número** es el ID: *cada ID tiene su propio color*, así un cambio de color en el mismo objeto = el tracker le cambió el ID.

In [ ]:
# ======================================================================
# 4.2 · fuse_moving + draw
# Qué hace : Une fragmentos que se mueven juntos y dibuja cajas e IDs
# ¿Se toca? : No
# ======================================================================
GREEN, RED, ORANGE = (60, 200, 60), (40, 40, 255), (0, 150, 255)


def fuse_moving(tracks, cfg):
    """Agrupa tracks en movimiento que están pegados y comparten velocidad (p. ej. las mitades de un camión)."""
    mv = [t for t in tracks if t.moving]
    parent = list(range(len(mv)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i

    cosmin = np.cos(np.deg2rad(cfg.merge_angle_deg))
    for i in range(len(mv)):
        for j in range(i + 1, len(mv)):
            a, b = mv[i], mv[j]
            va, vb = np.linalg.norm(a.vec), np.linalg.norm(b.vec)
            if va < 1e-6 or vb < 1e-6:
                continue
            if float(a.vec @ b.vec) / (va * vb) < cosmin or abs(va - vb) > cfg.merge_speed_ratio * max(va, vb):
                continue
            dx = max(0.0, max(a.box[0], b.box[0]) - min(a.box[2], b.box[2]))
            dy = max(0.0, max(a.box[1], b.box[1]) - min(a.box[3], b.box[3]))
            side = max(a.box[2] - a.box[0], a.box[3] - a.box[1], b.box[2] - b.box[0], b.box[3] - b.box[1])
            if np.hypot(dx, dy) <= cfg.merge_gap * side:
                parent[find(i)] = find(j)
    groups = defaultdict(list)
    for i, t in enumerate(mv):
        groups[find(i)].append(t)
    out = []
    for g in groups.values():
        if len(g) > 1 and all(t.cls < 0 for t in g):
            w = max(t.box[2] for t in g) - min(t.box[0] for t in g)
            h = max(t.box[3] for t in g) - min(t.box[1] for t in g)
            if max(w, h) / max(min(w, h), 1.0) > cfg.merge_max_aspect:
                out.extend([t] for t in g)           # cadena de fragmentos a lo largo de un cable/borde: no es un objeto
                continue
        out.append(g)
    return out


def id_color(tid):
    """Color estable por ID (BGR): el mismo ID siempre tiene el mismo color, otro ID otro color."""
    h = int((tid * 47) % 180)
    c = cv2.cvtColor(np.uint8([[[h, 220, 230]]]), cv2.COLOR_HSV2BGR)[0, 0]
    return int(c[0]), int(c[1]), int(c[2])


def id_badge(frame, x, y, tid):
    txt = f"{tid}"
    (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
    cv2.rectangle(frame, (x, y - th - 10), (x + tw + 10, y), id_color(tid), -1)
    cv2.putText(frame, txt, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2, cv2.LINE_AA)


def draw(frame, tracks, k, names, thr, cfg):
    for t in tracks:                                # estáticos: una caja por track
        if t.moving or t.cls < 0:
            continue
        x1, y1, x2, y2 = (t.box * k).astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)
        cv2.putText(frame, f"{names.get(t.cls, '?')}", (x1 + 36, max(14, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, GREEN, 1, cv2.LINE_AA)
        if cfg.id_badge:
            id_badge(frame, x1, max(24, y1), t.id)
    if cfg.show_parallax:
        for t in tracks:
            if t.par and not t.moving:
                x1, y1, x2, y2 = (t.box * k).astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (170, 170, 170), 1)
                cv2.putText(frame, "paralaje?", (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (170, 170, 170), 1, cv2.LINE_AA)
    groups = fuse_moving(tracks, cfg) if cfg.merge_motion else [[t] for t in tracks if t.moving]
    for g in groups:
        box = np.array([min(t.box[0] for t in g), min(t.box[1] for t in g),
                        max(t.box[2] for t in g), max(t.box[3] for t in g)])
        known = [t for t in g if t.cls >= 0]
        main = max(known or g, key=lambda t: (t.box[2] - t.box[0]) * (t.box[3] - t.box[1]))
        lab = names.get(main.cls, "?") if known else "desconocido"
        color = RED if known else ORANGE
        x1, y1, x2, y2 = (box * k).astype(int)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{lab} {np.mean([t.disp for t in g]):.0f}px",
                    (x1 + 36, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        if cfg.id_badge:
            id_badge(frame, x1, max(24, y1), min(t.id for t in g))
        if len(main.pts) > 1:
            cv2.polylines(frame, [(main.pts * k).astype(np.int32).reshape(-1, 1, 2)], False, color, 1)
    n_static = sum(1 for t in tracks if not t.moving and t.cls >= 0)
    cv2.putText(frame, f"En movimiento: {len(groups)} | Estaticos: {n_static} | umbral: {thr:.1f}px",
                (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

## PARTE 5 — Utilidades y ejecución
`real_fps` corrige el FPS que OpenCV lee mal en algunos videos. `buscar_tramos_dificiles` propone dónde se mueve más la cámara. `run()` procesa el video con el tracker de este notebook y guarda video, CSV de tracks y CSV de métricas (mismas columnas que el benchmark).

In [ ]:
# ======================================================================
# 5 · real_fps + resumen + métricas + buscar_tramos_dificiles + run
# Qué hace : Utilidades y ejecución completa con el tracker de ESTE notebook
# ¿Se toca? : No
# ======================================================================
def real_fps(video):
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps and 1 < fps <= 240:
        cap.release(); return fps
    n, last = 0, 0                                 # fps absurdo (p. ej. 1000 en algunos .webm): se mide con timestamps
    while cap.grab():
        n += 1; last = cap.get(cv2.CAP_PROP_POS_MSEC)
    cap.release()
    return n / (last / 1000.0) if last > 0 else 30.0


def _rows(history, names):
    rows = []
    for t in history.values():
        rows.append(dict(id=t.id, clase=names.get(t.cls, "desconocido") if t.cls >= 0 else "desconocido",
                         primer_frame=t.first, ultimo_frame=t.last, detecciones=t.hits,
                         frames_en_movimiento=t.moving_frames, frames_paralaje=t.par_frames,
                         cambios_de_estado=t.toggles, desplazamiento_max_px=round(t.max_disp, 1),
                         estado_final="movimiento" if t.moving else "estatico"))
    return rows


def _metrics(kind, history, n_frames, fps, shown_counts, raw_n=0):
    hs = list(history.values())
    if not hs:                                   # ningún track llegó a confirmarse (p. ej. el tracker no logra seguir nada)
        return dict(variante=kind, ids_emitidos=raw_n, tracks=0, ids_por_minuto=0.0, duracion_mediana_s=0.0, tracks_menores_1s_pct=0.0,
                    objetos_simultaneos_prom=0.0, pct_frames_en_movimiento=0.0, cambios_estado_por_track=0.0)
    lens = np.array([t.hits for t in hs])
    minutos = max(n_frames / fps / 60.0, 1e-9)
    return dict(variante=kind,
                ids_emitidos=raw_n,
                tracks=len(hs),
                ids_por_minuto=round(len(hs) / minutos, 1),
                duracion_mediana_s=round(float(np.median(lens)) / fps, 2),
                tracks_menores_1s_pct=round(100.0 * float(np.mean(lens < fps)), 1),
                objetos_simultaneos_prom=round(float(np.mean(shown_counts)), 2),
                pct_frames_en_movimiento=round(100.0 * sum(t.moving_frames for t in hs) / max(int(lens.sum()), 1), 1),
                cambios_estado_por_track=round(float(np.mean([t.toggles for t in hs])), 2))


def buscar_tramos_dificiles(video, ventana_s=10.0, top=3, work_width=480, paso=2):
    """Escanea el video y propone los tramos donde la CÁMARA se mueve más rápido.
    Es donde los trackers se diferencian: con la cámara casi quieta, todos asocian igual.
    Devuelve una lista de dicts con 'start' (frame) y 'max_frames' listos para compare()."""
    cfg = Config(work_width=work_width)
    fps = real_fps(video)
    cap = cv2.VideoCapture(str(video))
    W0, H0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ww, wh = work_width, int(round(H0 * work_width / W0))
    gm = GlobalMotion(cfg)
    centro = np.array([[ww / 2, wh / 2]], np.float32)
    vel, frames = [], []
    i = 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if i % paso == 0:
            g = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(f, (ww, wh), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY), (3, 3), 0)
            H, ok_h = gm.step(g)
            d = np.linalg.norm(warp_pts(H, centro) - centro) if ok_h else 0.0
            vel.append(d * fps / paso * (W0 / ww))            # px/s en la resolución ORIGINAL
            frames.append(i)
        i += 1
    cap.release()
    vel = np.array(vel)
    n_v = max(2, int(round(ventana_s * fps / paso)))
    if len(vel) < n_v:
        n_v = len(vel)
    media = np.convolve(vel, np.ones(n_v) / n_v, mode="valid")
    orden = np.argsort(-media)
    elegidos = []
    for j in orden:
        if all(abs(j - e) >= n_v for e in elegidos):
            elegidos.append(j)
        if len(elegidos) >= top:
            break
    print(f"Velocidad de cámara (px/s en resolución original) | mediana global = {np.median(vel):.0f}")
    out = []
    for j in elegidos:
        start = frames[j]
        r = dict(start=int(start), max_frames=int(ventana_s * fps), t_inicio_s=round(start / fps, 1),
                 vel_media_px_s=round(float(media[j]), 0), vel_max_px_s=round(float(vel[j:j + n_v].max()), 0))
        out.append(r)
        print(f"  tramo: start={r['start']:5d} (t={r['t_inicio_s']:6.1f} s)  vel media={r['vel_media_px_s']:6.0f}  máx={r['vel_max_px_s']:6.0f}")
    return out


def run(video, detector=None, cfg=None, out_dir=None, start=0, max_frames=None, names=None, **kw_tracker):
    """Procesa el video con el tracker de ESTE notebook (TrackerSistema).
    Salida en out_dir: <nombre>.mp4, tracks_<nombre>.csv, metricas_<nombre>.csv (y <nombre>_residuo.mp4 si debug_residual).
    Las métricas tienen exactamente las mismas columnas que comparativa_metricas.csv del benchmark."""
    cfg = cfg or Config()
    names = names or (getattr(detector, "names", None) or {})
    out_dir = out_dir or os.path.join("salidas", TrackerSistema.nombre)
    os.makedirs(out_dir, exist_ok=True)
    fps = real_fps(video)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {video}")
    W0, H0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = cfg.work_width / W0                    # original -> trabajo
    ww, wh = cfg.work_width, int(round(H0 * scale))
    ow = cfg.out_width or W0
    oh = int(round(H0 * ow / W0))
    k_out = ow / ww                                # trabajo -> salida
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    nombre = TrackerSistema.nombre
    writer = cv2.VideoWriter(os.path.join(out_dir, f"{nombre}.mp4"), fourcc, fps, (ow, oh))
    writer_dbg = cv2.VideoWriter(os.path.join(out_dir, f"{nombre}_residuo.mp4"), fourcc, fps, (ow, oh)) if cfg.debug_residual else None

    skip_ids = [i for i, n in names.items() if str(n).lower() in cfg.parallax_skip]
    gm, ms = GlobalMotion(cfg), MotionState(cfg, fps, skip_ids)
    trk = TrackerSistema(cfg, fps, **kw_tracker)
    prop = MotionProposals(cfg, fps) if cfg.use_proposals else None
    history, counts, i, bad = {}, [], 0, 0
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames is not None and i >= max_frames):
            break
        small = cv2.resize(frame, (ww, wh), interpolation=cv2.INTER_AREA)
        gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (3, 3), 0)
        H, h_ok = gm.step(gray)
        bad += int(not h_ok)

        dets = detector(frame, scale) if detector is not None else np.zeros((0, 6), np.float32)
        if len(dets):                                # cajas gigantes = falsos positivos (casas, terrenos)
            area = (dets[:, 2] - dets[:, 0]) * (dets[:, 3] - dets[:, 1])
            dets = dets[area <= cfg.max_box_area_frac * ww * wh]
        if prop is not None:
            pr = prop.step(gray, H, h_ok)
            if len(pr) and len(dets):                # si YOLO ya explica el cambio, la propuesta sobra
                pr = pr[iou_pad(pr[:, :4], dets[:, :4], 2.0).max(1) < 0.1]
            if len(pr):
                dets = np.vstack([dets, pr]) if len(dets) else pr

        shown = trk.update(dets, H, h_ok, start + i, small)
        ms.update(shown, h_ok)
        for t in shown:
            history[t.id] = t
        counts.append(len(shown))
        vis = cv2.resize(frame, (ow, oh), interpolation=cv2.INTER_AREA)
        draw(vis, shown, k_out, names, ms.thr, cfg)
        cv2.rectangle(vis, (0, oh - 60), (ow, oh), (0, 0, 0), -1)
        cv2.putText(vis, f"{TrackerSistema.titulo} | frame {start + i} | IDs emitidos: {trk.ids_emitidos}",
                    (20, oh - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(vis)
        if writer_dbg is not None and prop is not None and prop.last_diff is not None:
            res = cv2.cvtColor(cv2.convertScaleAbs(prop.last_diff, alpha=8), cv2.COLOR_GRAY2BGR)
            res = cv2.resize(res, (ow, oh))
            draw(res, shown, k_out, names, ms.thr, cfg)
            writer_dbg.write(res)
        i += 1
        if i % 100 == 0:
            print(f"{i} fotogramas")
    cap.release(); writer.release()
    if writer_dbg is not None:
        writer_dbg.release()

    rows = _rows(history, names)
    with open(os.path.join(out_dir, f"tracks_{nombre}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["id"])
        w.writeheader(); w.writerows(rows)
    m = _metrics(nombre, history, i, fps, counts or [0], trk.ids_emitidos)
    with open(os.path.join(out_dir, f"metricas_{nombre}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(m.keys())); w.writeheader(); w.writerow(m)
    print(f"FPS={fps:.2f} | frames={i} | frames sin homografía fiable={bad} | tracks={len(rows)} | salida en {out_dir}/")
    return m, rows

## PARTE 6 — Ejecución
`PARTE 1` → … → `PARTE 5` solo definen cosas. Aquí se ejecuta. Para probar rápido usa `max_frames=300`; `start` permite empezar en un tramo (p. ej. `start=1500` = tramo final de `input1.mp4`, donde el dron desciende sobre el poste).

### Paso A — Verificar Ultralytics (ejecutar primero)
Si falla, imprime un diagnóstico de tu versión.

In [ ]:
GMC_METHOD = "sparseOptFlow"   # "sparseOptFlow" | "orb" | "ecc" | "none" (sin compensar cámara)
selftest_ultralytics(Config(), fps=30.0)

### Paso B — Procesar el video

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Video, display

VIDEO_PATH = Path("input1.mp4")      # <-- CAMBIAR por tu video
OUT_DIR = "salidas/botsort"

cfg = Config(work_width=960, out_width=1920)
detector = YoloDetector(
    "yolo11n-obb.pt", imgsz=1280, conf=0.15,
    class_names=("small vehicle", "large vehicle"),   # solo vehículos: el modelo DOTA también "ve" barcos, puentes, puertos…
)

metricas, filas = run(VIDEO_PATH, detector, cfg, out_dir=OUT_DIR, start=0, max_frames=None, gmc_method=GMC_METHOD)

display(pd.DataFrame([metricas]).set_index("variante"))
display(pd.DataFrame(filas).sort_values("frames_en_movimiento", ascending=False).head(15))

In [ ]:
display(Video(f"{OUT_DIR}/botsort.mp4", embed=False, width=1100))